In [1]:
import numpy as np
import pandas as pd
from collections import defaultdict

In [7]:
class PhonemeHMM:
    def __init__(self):
        # Define the phonemes
        self.phonemes = ['/s/', '/p/', '/ie:/', '/tS/']
        self.observations = ['Energy', 'Pitch', 'Duration']

        # Initial Probabilities (probability of starting with each phoneme)
        self.initial_probs = {
            '/s/': 1.0,
            '/p/': 0.0,
            '/ie:/': 0.0,
            '/tS/': 0.0
        }

        # Transition Probabilities (from one phoneme to another)
        self.transition_probs = {
            '/s/': {'/s/': 0.0, '/p/': 0.9, '/ie:/': 0.0, '/tS/': 0.1},
            '/p/': {'/s/': 0.0, '/p/': 0.0, '/ie:/': 0.9, '/tS/': 0.1},
            '/ie:/': {'/s/': 0.0, '/p/': 0.0, '/ie:/': 0.0, '/tS/': 1.0},
            '/tS/': {'/s/': 0.0, '/p/': 0.0, '/ie:/': 0.0, '/tS/': 0.0}
        }

        # Emission Probabilities (probability of observation given phoneme)
        self.emission_probs = {
            '/s/': {'Energy': 0.7, 'Pitch': 0.2, 'Duration': 0.1},
            '/p/': {'Energy': 0.5, 'Pitch': 0.3, 'Duration': 0.2},
            '/ie:/': {'Energy': 0.3, 'Pitch': 0.5, 'Duration': 0.2},
            '/tS/': {'Energy': 0.4, 'Pitch': 0.4, 'Duration': 0.2}
        }

    # ========================================================================
    # Task (b): Display HMM matrices neatly
    # ========================================================================

    def display_hmm_parameters(self):

        print("HMM PARAMETERS FOR PHONEME RECOGNITION (Word: 'SPEECH')")


        # Display Initial Probabilities
        print("\n1. INITIAL PROBABILITIES")

        initial_df = pd.DataFrame(list(self.initial_probs.items()),
                                  columns=['Phoneme', 'Probability'])
        print(initial_df.to_string(index=False))

        # Display Transition Probabilities Matrix
        print("\n2. TRANSITION PROBABILITIES MATRIX")

        trans_matrix = pd.DataFrame(self.transition_probs).T
        print(trans_matrix.to_string())

        # Display Emission Probabilities Matrix
        print("\n3. EMISSION PROBABILITIES MATRIX")

        emis_matrix = pd.DataFrame(self.emission_probs).T
        print(emis_matrix.to_string())
        print("=" * 80 + "\n")

    # ========================================================================
    # Task (c): Generate sequence of phonemes and observations
    # ========================================================================

    def generate_sequence(self):
        """Generate a sequence of phonemes and corresponding observations"""

        # Start with initial phoneme (always /s/)
        current_phoneme = '/s/'
        phoneme_sequence = [current_phoneme]
        observation_sequence = []

        # Generate observations for the starting phoneme
        obs = np.random.choice(
            self.observations,
            p=[self.emission_probs[current_phoneme][obs]
               for obs in self.observations]
        )
        observation_sequence.append(obs)

        # Continue generating until we reach /tS/ (end phoneme)
        while current_phoneme != '/tS/':
            # Get transition probabilities from current phoneme
            trans_probs = list(self.transition_probs[current_phoneme].values())

            # Select next phoneme based on transition probabilities
            next_phoneme = np.random.choice(
                self.phonemes,
                p=trans_probs
            )

            phoneme_sequence.append(next_phoneme)
            current_phoneme = next_phoneme

            # Generate observation for this phoneme
            obs = np.random.choice(
                self.observations,
                p=[self.emission_probs[current_phoneme][obs]
                   for obs in self.observations]
            )
            observation_sequence.append(obs)

        return phoneme_sequence, observation_sequence

    # ========================================================================
    # Task (d): Viterbi Algorithm for Inference (Finding most likely sequence)
    # ========================================================================

    def viterbi_inference(self, observations):
        """
        Viterbi Algorithm: Find the most likely sequence of phonemes
        given the observed acoustic features
        """
        n_phonemes = len(self.phonemes)
        n_observations = len(observations)

        # Initialize Viterbi matrix and path matrix
        viterbi_matrix = np.zeros((n_phonemes, n_observations))
        path_matrix = np.zeros((n_phonemes, n_observations), dtype=int)

        # Initialization: First observation
        for i, phoneme in enumerate(self.phonemes):
            viterbi_matrix[i, 0] = self.initial_probs[phoneme] * \
                                   self.emission_probs[phoneme][observations[0]]

        # Recursion: Forward pass
        for t in range(1, n_observations):
            for j, phoneme_j in enumerate(self.phonemes):
                # Calculate probability for all possible previous phonemes
                probs = []
                for i, phoneme_i in enumerate(self.phonemes):
                    prob = viterbi_matrix[i, t-1] * \
                           self.transition_probs[phoneme_i][phoneme_j] * \
                           self.emission_probs[phoneme_j][observations[t]]
                    probs.append(prob)

                # Store maximum probability and path
                viterbi_matrix[j, t] = max(probs)
                path_matrix[j, t] = np.argmax(probs)

        # Backtracking: Find the best path
        best_path_indices = [np.argmax(viterbi_matrix[:, n_observations-1])]

        for t in range(n_observations-1, 0, -1):
            best_path_indices.insert(0, path_matrix[best_path_indices[0], t])

        # Convert indices to phoneme sequence
        best_phoneme_sequence = [self.phonemes[i] for i in best_path_indices]

        return best_phoneme_sequence, viterbi_matrix

    def display_viterbi_results(self, observations, best_sequence, viterbi_matrix):
        """Display Viterbi inference results"""
        print("VITERBI INFERENCE RESULTS")
        print("=" * 80)
        print(f"\nObserved Acoustic Features: {observations}")
        print(f"Inferred Phoneme Sequence: {best_sequence}")

        print("\nViterbi Probability Matrix:")
        print("-" * 40)
        viterbi_df = pd.DataFrame(viterbi_matrix,
                                  index=self.phonemes,
                                  columns=[f"Obs {i+1}\n({observations[i]})"
                                          for i in range(len(observations))])
        print(viterbi_df.to_string())

In [9]:
if __name__ == "__main__":
    # Initialize HMM
    hmm = PhonemeHMM()

    # Task (b): Display HMM parameters
    hmm.display_hmm_parameters()

    # Task (c): Generate sequence of phonemes and observations
    print("GENERATING PHONEME AND OBSERVATION SEQUENCES")


    phoneme_seq, observation_seq = hmm.generate_sequence()

    print(f"Generated phoneme sequence: {phoneme_seq}")
    print(f"Corresponding observations:  {observation_seq}")


    # Task (d): Viterbi Inference - Decode the observation sequence
    print("PERFORMING VITERBI INFERENCE")


    # Use the generated observation sequence for inference
    best_sequence, viterbi_matrix = hmm.viterbi_inference(observation_seq)
    hmm.display_viterbi_results(observation_seq, best_sequence, viterbi_matrix)

    # Additional inference example with a different observation sequence
    test_observations = ['Energy', 'Pitch', 'Duration', 'Energy']
    best_sequence_test, viterbi_matrix_test = hmm.viterbi_inference(test_observations)
    hmm.display_viterbi_results(test_observations, best_sequence_test, viterbi_matrix_test)

HMM PARAMETERS FOR PHONEME RECOGNITION (Word: 'SPEECH')

1. INITIAL PROBABILITIES
Phoneme  Probability
    /s/          1.0
    /p/          0.0
  /ie:/          0.0
   /tS/          0.0

2. TRANSITION PROBABILITIES MATRIX
       /s/  /p/  /ie:/  /tS/
/s/    0.0  0.9    0.0   0.1
/p/    0.0  0.0    0.9   0.1
/ie:/  0.0  0.0    0.0   1.0
/tS/   0.0  0.0    0.0   0.0

3. EMISSION PROBABILITIES MATRIX
       Energy  Pitch  Duration
/s/       0.7    0.2       0.1
/p/       0.5    0.3       0.2
/ie:/     0.3    0.5       0.2
/tS/      0.4    0.4       0.2

GENERATING PHONEME AND OBSERVATION SEQUENCES
Generated phoneme sequence: ['/s/', np.str_('/p/'), np.str_('/ie:/'), np.str_('/tS/')]
Corresponding observations:  [np.str_('Energy'), np.str_('Pitch'), np.str_('Energy'), np.str_('Pitch')]
PERFORMING VITERBI INFERENCE
VITERBI INFERENCE RESULTS

Observed Acoustic Features: [np.str_('Energy'), np.str_('Pitch'), np.str_('Energy'), np.str_('Pitch')]
Inferred Phoneme Sequence: ['/s/', '/p/', '/ie: